<a href="https://colab.research.google.com/github/Vigneshvar16/CSA0607-DAA/blob/main/CO4_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DAA CO4B - Vigneshvar S (192424187) |
import heapq, itertools, math, random, time, tracemalloc
from functools import cmp_to_key

RUN = [1, 2, 3, 4, 5, 6]            # choose which tasks to run
INF = math.inf
def banner(t): print("\n" + "=" * 72 + "\n" + t + "\n" + "=" * 72)

# ============================ TASK 1: WORD WRAP (DP) ============================
def word_wrap(words, width):
    """best[i] = min penalty for laying out words i..n-1 when a new line starts at word i.
       nxt[i]  = index of the first word of the following line (the choice that gave best[i])."""
    n = len(words)
    if n == 0: return 0, [], [0], [0]
    if max(map(len, words)) > width: raise ValueError(f"a word is longer than the line width {width}")
    best, nxt = [0] * (n + 1), [n] * (n + 1)
    for i in range(n - 1, -1, -1):
        best[i], used = INF, -1
        for j in range(i, n):                          # candidate line = words i..j
            used += len(words[j]) + 1                  # characters used incl. single spaces
            if used > width: break
            cost = 0 if j == n - 1 else (width - used) ** 2 + best[j + 1]   # final line is free
            if cost <= best[i]: best[i], nxt[i] = cost, j + 1
    lines, i = [], 0
    while i < n:
        j = nxt[i]; text = " ".join(words[i:j]); unused = width - len(text)
        lines.append((text, unused, 0 if j == n else unused ** 2)); i = j
    assert sum(p for _, _, p in lines) == best[0]
    return best[0], lines, best, nxt

def show_wrap(text, width, show_states=False):
    words = text.split(); total, lines, best, nxt = word_wrap(words, width)
    print(f"\nWidth {width}: total penalty = {total}, lines = {len(lines)}")
    for k, (t, u, p) in enumerate(lines, 1): print(f"  line {k}: |{t:<{width}}| unused={u} penalty={p}")
    if show_states:
        print("  DP states  i | word | best[i] | next line starts at word")
        for i, w in enumerate(words): print(f"  {i:>10} | {w:<11}| {best[i]:>7} | {nxt[i]}")
    return total, lines

def task1():
    banner("TASK 1 - WORD WRAP")
    text = "Dynamic programming stores answers to small overlapping subproblems so each is solved once"
    print("Text:", text)
    show_wrap(text, 20, show_states=True)
    print("\n-- 1.3 requirement change (same text, other widths) --")
    pred = {14: "line 1 changes first (only 'Dynamic' fits); more lines; total penalty increases",
            26: "line 1 changes first ('stores' now fits exactly); fewer lines; total penalty decreases",
            40: "line 1 changes first (more words fit); fewer lines; total penalty decreases"}
    for w in (14, 26, 40):
        print("\nPrediction for width", w, "->", pred[w]); show_wrap(text, w)
    print("\n-- 1.4 boundary tests --")
    print("width == longest word (11):", word_wrap(text.split(), 11)[0])
    try: word_wrap(text.split(), 10)
    except ValueError as e: print("width 10 ->", "ValueError:", e)
    print("whole text fits one line (width 100):", word_wrap(text.split(), 100)[0])
    print("single word:", word_wrap(["hello"], 5)[0], "| empty text:", word_wrap([], 5)[0])

# ============================ TASK 2: GREEDY vs DP ============================
def sort_by_ratio(items, ops):
    def cmp(a, b):
        ops["comparisons"] += 1; ra, rb = a[2] / a[1], b[2] / b[1]
        return (rb > ra) - (rb < ra)
    return sorted(items, key=cmp_to_key(cmp))

def fractional_greedy(items, cap, verbose=False):
    ops = {"comparisons": 0, "scan steps": 0}; rem, total, picks = cap, 0.0, []
    for iid, w, v in sort_by_ratio(items, ops):
        if rem <= 0: break
        ops["scan steps"] += 1; f = min(1.0, rem / w); picks.append((iid, round(f, 3)))
        total += f * v; rem -= f * w
        if verbose: print(f"   take {iid}: ratio {v/w:.2f}, fraction {f:.2f}, remaining capacity {rem:.1f}, value so far {total:.1f}")
    ops["memory cells"] = len(items); return total, picks, ops

def greedy_01(items, cap, verbose=False):
    ops = {"comparisons": 0, "scan steps": 0}; rem, total, picks = cap, 0, []
    for iid, w, v in sort_by_ratio(items, ops):
        ops["scan steps"] += 1
        if w <= rem:
            picks.append(iid); total += v; rem -= w
            if verbose: print(f"   commit {iid}: ratio {v/w:.2f}, remaining capacity {rem}, value so far {total}")
        elif verbose: print(f"   skip {iid}: weight {w} > remaining {rem}")
    ops["memory cells"] = len(items); return total, picks, ops

def knapsack_dp(items, cap):
    n = len(items); dp = [[0] * (cap + 1) for _ in range(n + 1)]; ops = {"cells computed": 0}
    for i in range(1, n + 1):
        _, w, v = items[i - 1]
        for c in range(cap + 1):
            ops["cells computed"] += 1; dp[i][c] = dp[i - 1][c]
            if w <= c: dp[i][c] = max(dp[i][c], v + dp[i - 1][c - w])
    chosen, c = [], cap
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i - 1][c]: chosen.append(items[i - 1][0]); c -= items[i - 1][1]
    ops["memory cells"] = (n + 1) * (cap + 1); return dp[n][cap], chosen[::-1], ops

def task2():
    banner("TASK 2 - GREEDY vs DYNAMIC PROGRAMMING")
    normal = [("A", 10, 60), ("B", 20, 100), ("C", 20, 70), ("D", 15, 15)]
    print("2.1 normal case, capacity 50:", normal)
    print(" Fractional greedy decision sequence:"); t, p, _ = fractional_greedy(normal, 50, True)
    print(" ->", p, "value", t); print(" 0/1 greedy:", greedy_01(normal, 50)[:2], "| 0/1 DP:", knapsack_dp(normal, 50)[:2])
    bad = [("A", 6, 30), ("B", 5, 24), ("C", 5, 24)]
    print("\n2.2 my 0/1 counterexample, capacity 10:", bad)
    print(" 0/1 greedy decisions:"); g = greedy_01(bad, 10, True); d = knapsack_dp(bad, 10)
    print(f" GreedyResult = {g[0]} {g[1]}   DPOptimum = {d[0]} {d[1]}   GreedyResult != DPOptimum: {g[0] != d[0]}")
    print(" First greedy commitment preventing optimum: take A (best ratio 5.0), leaving 4 units where nothing fits.")
    print("\n2.3 engineering comparison on a larger instance (n=200, capacity 1000, seed 7)")
    rnd = random.Random(7); big = [(f"I{i}", rnd.randint(5, 60), rnd.randint(10, 100)) for i in range(200)]
    gf = fractional_greedy(big, 1000); g1 = greedy_01(big, 1000); dd = knapsack_dp(big, 1000)
    print(f" Fractional greedy: value {gf[0]:.1f}  ops {gf[2]}")
    print(f" 0/1 greedy       : value {g1[0]}  ops {g1[2]}")
    print(f" 0/1 DP           : value {dd[0]}  ops {dd[2]}")
    print(f" 0/1 greedy quality = {100 * g1[0] / dd[0]:.2f}% of the DP optimum")
    ratios, opt = [], 0
    for s in range(300):
        r = random.Random(s); its = [(f"I{i}", r.randint(1, 30), r.randint(1, 50)) for i in range(15)]
        cap = sum(w for _, w, _ in its) // 3; a = greedy_01(its, cap)[0]; b = knapsack_dp(its, cap)[0]
        ratios.append(a / b); opt += (a == b)
    print(f" 300 random small instances: greedy optimal in {opt}, mean quality {100*sum(ratios)/300:.2f}%, worst {100*min(ratios):.2f}%")

# ============================ TASK 3: ROUTING ENGINE ============================
NODES = list("SABCDEFG")
BASE = [("S","A",5),("S","B",2),("A","B",1),("B","C",4),("A","D",3),("C","D",2),("B","E",7),
        ("C","E",3),("D","F",4),("E","F",1),("D","G",6),("F","G",2),("E","G",5)]   # 8 vertices, 13 edges

def dijkstra(nodes, edges, src):
    adj = {v: [] for v in nodes}
    for u, v, w in edges: adj[u].append((v, w))
    dist = {v: INF for v in nodes}; pred = {v: None for v in nodes}; dist[src] = 0
    heap, final, order = [(0, src)], set(), []
    while heap:
        d, u = heapq.heappop(heap)
        if u in final: continue
        final.add(u); order.append(u)                 # greedy commitment: dist[u] is never changed again
        for v, w in adj[u]:
            if v not in final and d + w < dist[v]:
                dist[v] = d + w; pred[v] = u; heapq.heappush(heap, (dist[v], v))
    return dist, pred, order

def path_to(pred, src, v):
    if v != src and pred[v] is None: return None
    p = [v]
    while p[-1] != src: p.append(pred[p[-1]])
    return p[::-1]

def bellman_ford(nodes, edges, src):
    dist = {v: INF for v in nodes}; pred = {v: None for v in nodes}; dist[src] = 0; passes = 0
    for _ in range(len(nodes) - 1):
        changed = False; passes += 1
        for u, v, w in edges:
            if dist[u] != INF and dist[u] + w < dist[v]: dist[v] = dist[u] + w; pred[v] = u; changed = True
        if not changed: break
    for u, v, w in edges:                              # extra pass: any relaxation => reachable negative cycle
        if dist[u] != INF and dist[u] + w < dist[v]:
            pred[v] = u; x = v
            for _ in nodes: x = pred[x]                # step back |V| times to land inside the cycle
            cyc = [x]; y = pred[x]
            while y != x: cyc.append(y); y = pred[y]
            return dist, pred, passes, cyc[::-1] + [cyc[-1]]
    return dist, pred, passes, None

def report(title, dist, pred, src, order=None):
    print(f"\n{title}")
    if order: print("  finalised order:", " -> ".join(order))
    for v in NODES: print(f"  {src}->{v}: dist={dist[v]!s:>4} pred={pred[v]!s:>4} path={'-'.join(path_to(pred, src, v) or ['unreachable'])}")

def floyd(nodes, edges):
    D = {i: {j: (0 if i == j else INF) for j in nodes} for i in nodes}
    for u, v, w in edges: D[u][v] = min(D[u][v], w)
    for k in nodes:
        for i in nodes:
            for j in nodes:
                if D[i][k] + D[k][j] < D[i][j]: D[i][j] = D[i][k] + D[k][j]
    return D

def warshall(nodes, edges):
    R = {i: {j: i == j for j in nodes} for i in nodes}
    for u, v, _ in edges: R[u][v] = True
    for k in nodes:
        for i in nodes:
            for j in nodes: R[i][j] = R[i][j] or (R[i][k] and R[k][j])
    return R

def matrix(M, nodes, fmt):
    print("      " + "".join(f"{v:>5}" for v in nodes))
    for i in nodes: print(f"  {i:>3} " + "".join(f"{fmt(M[i][j]):>5}" for j in nodes))

def task3():
    banner("TASK 3 - ROUTING ENGINE")
    d, p, o = dijkstra(NODES, BASE, "S"); report("Stage A: Dijkstra from S (non-negative graph)", d, p, "S", o)
    neg = BASE + [("C", "A", -3)]
    print("\nStage B prediction: with edge C->A = -3 Dijkstra becomes INVALID. A is finalised at 5 (direct S->A) before C"
          " (dist 6) is processed, but S->B->C->A costs 2+4-3 = 3. Routes through A (A, D, F, G) will be affected.")
    d2, p2, o2 = dijkstra(NODES, neg, "S"); report("Dijkstra on modified graph", d2, p2, "S", o2)
    bd, bp, passes, cyc = bellman_ford(NODES, neg, "S"); report(f"Bellman-Ford on modified graph ({passes} passes)", bd, bp, "S")
    print("  Vertices where Dijkstra is wrong:", [v for v in NODES if d2[v] != bd[v]])
    cyc_graph = BASE + [("C", "A", -6)]
    print("\nStage C: C->A = -6 makes cycle A->B->C->A cost 1+4-6 = -1")
    dc, pc, _, cyc = bellman_ford(NODES, cyc_graph, "S")
    if cyc: print("  NEGATIVE CYCLE DETECTED, shortest paths undefined. Cycle:", " -> ".join(cyc))
    print("\nStage D: all pairs on the Stage B graph (negative edge, no negative cycle)")
    print(" Floyd distance matrix:"); matrix(floyd(NODES, neg), NODES, lambda x: "INF" if x == INF else str(x))
    print(" Warshall reachability matrix:"); matrix(warshall(NODES, neg), NODES, lambda x: "1" if x else "0")

# ============================ TASK 4: ASSEMBLY LINE ============================
def assembly_line(a, t, e, x, verbose=False):
    """a[l][i] station time; t[l][i] transfer time from line l after station i; e[l] entry; x[l] exit."""
    n = len(a[0])
    if n == 0 or len(a[1]) != n or any(len(t[l]) != n - 1 for l in (0, 1)): raise ValueError("inconsistent input sizes")
    f = [[0] * n for _ in range(2)]; came = [[None] * n for _ in range(2)]; rows = []
    for l in (0, 1): f[l][0] = e[l] + a[l][0]
    for i in range(1, n):
        row = [i + 1]
        for l in (0, 1):
            o = 1 - l; stay = f[l][i - 1] + a[l][i]; move = f[o][i - 1] + t[o][i - 1] + a[l][i]
            f[l][i], came[l][i] = (stay, l) if stay <= move else (move, o); row += [stay, move, "stay" if stay <= move else "transfer"]
        rows.append(row)
    fin = [f[l][n - 1] + x[l] for l in (0, 1)]; line = 0 if fin[0] <= fin[1] else 1; total = fin[line]; route = [line]
    for i in range(n - 1, 0, -1): line = came[line][i]; route.append(line)
    return total, route[::-1], rows

def show_route(total, route):
    print("  Entry")
    for i, l in enumerate(route):
        if i and route[i - 1] != l: print("   | Transfer")
        else: print("   |")
        print(f"  Line {l + 1} - Station {i + 1}")
    print("   |\n  Exit\n  Minimum Production Time =", total)

def task4():
    banner("TASK 4 - ASSEMBLY LINE SCHEDULING")
    a = [[7, 9, 3, 4, 8, 4], [8, 5, 6, 4, 5, 7]]; t = [[2, 3, 1, 3, 4], [2, 1, 2, 2, 1]]; e = [2, 4]; x = [3, 2]
    total, route, rows = assembly_line(a, t, e, x); print("Base data: a =", a, "t =", t, "e =", e, "x =", x)
    print("\nDP decisions (cost to reach each line at each station):")
    print("  Stn | L1 stay | L1 transfer | chosen   | L2 stay | L2 transfer | chosen")
    for r in rows: print(f"  {r[0]:>3} | {r[1]:>7} | {r[2]:>11} | {r[3]:<8} | {r[4]:>7} | {r[5]:>11} | {r[6]}")
    show_route(total, route)
    t2 = [row[:] for row in t]; t2[0][0] = 20
    print("\n4.3 Prediction: raising transfer L1->L2 after station 1 from 2 to 20 is ON the optimal route, so the route will change"
          " (station 2 reached by staying on Line 1) and the total time will increase.")
    tt, rr, _ = assembly_line(a, t2, e, x); show_route(tt, rr)
    a2 = [a[0][:], a[1][:]]; a2[1][3] = 20
    print("\n4.4 Disruption: station 4 on Line 2 slows from 4 to 20")
    td, rd, _ = assembly_line(a2, t, e, x); show_route(td, rd)
    print("  old route lines:", [l + 1 for l in route], " new route lines:", [l + 1 for l in rd])
    try: assembly_line([[1, 2], [3]], [[1], [1]], [1, 1], [1, 1])
    except ValueError as err: print("  invalid input test ->", err)

# ============================ TASK 5: MST (KRUSKAL, PRIM, BORUVKA) ============================
V5 = list("ABCDEFG")
G5 = [("A","B",7),("A","D",5),("B","C",8),("B","D",9),("B","E",7),("C","E",5),("D","E",15),
      ("D","F",6),("E","F",8),("E","G",9),("F","G",11)]
G5_TIE = [(u, v, 7 if (u, v) == ("B", "D") else w) for u, v, w in G5]       # B-D lowered 9 -> 7 creates ties

def kruskal(vertices, edges, last=False, verbose=True):
    parent = {v: v for v in vertices}; finds = [0]
    def find(v):
        finds[0] += 1
        while parent[v] != v: parent[v] = parent[parent[v]]; v = parent[v]
        return v
    order = sorted(range(len(edges)), key=lambda k: (edges[k][2], -k if last else k)); mst = []
    for k in order:
        u, v, w = edges[k]; ru, rv = find(u), find(v)
        if ru != rv: parent[ru] = rv; mst.append(edges[k]); verbose and print(f"   accept {u}-{v} ({w})")
        else: verbose and print(f"   reject {u}-{v} ({w}) would form a cycle")
        if len(mst) == len(vertices) - 1: break
    return mst, {"find calls": finds[0], "edges sorted": len(edges)}

def prim(vertices, edges, start, verbose=True):
    adj = {v: [] for v in vertices}
    for k, (u, v, w) in enumerate(edges): adj[u].append((w, k, v, u)); adj[v].append((w, k, u, v))
    seen = {start}; heap = list(adj[start]); heapq.heapify(heap); mst = []; pushes = len(heap)
    while heap and len(mst) < len(vertices) - 1:
        w, k, v, u = heapq.heappop(heap)
        if v in seen: continue
        seen.add(v); mst.append(edges[k]); verbose and print(f"   add vertex {v} via {u}-{v} ({w})")
        for item in adj[v]:
            if item[2] not in seen: heapq.heappush(heap, item); pushes += 1
    return mst, {"heap pushes": pushes}

def boruvka(vertices, edges, verbose=True):
    comp = {v: v for v in vertices}; mst = []; rounds = 0
    while len(set(comp.values())) > 1:
        rounds += 1; cheapest = {}
        for k, (u, v, w) in enumerate(edges):
            cu, cv = comp[u], comp[v]
            if cu == cv: continue
            for c in (cu, cv):
                if c not in cheapest or (w, k) < (edges[cheapest[c]][2], cheapest[c]): cheapest[c] = k
        for k in sorted(set(cheapest.values())):
            u, v, w = edges[k]
            if comp[u] != comp[v]:
                mst.append(edges[k]); old, new = comp[u], comp[v]
                for z in comp:
                    if comp[z] == old: comp[z] = new
        verbose and print(f"   round {rounds}: cheapest edges {[edges[k][:2] + (edges[k][2],) for k in sorted(set(cheapest.values()))]}"
                          f" -> {len(set(comp.values()))} component(s)")
    return mst, {"rounds": rounds}

def edge_set(mst): return frozenset(frozenset((u, v)) for u, v, _ in mst)
def cost(mst): return sum(w for _, _, w in mst)

def all_msts(vertices, edges):
    best, found = INF, []
    for combo in itertools.combinations(range(len(edges)), len(vertices) - 1):
        par = {v: v for v in vertices}; ok = True
        def find(v):
            while par[v] != v: v = par[v]
            return v
        for k in combo:
            a, b = find(edges[k][0]), find(edges[k][1])
            if a == b: ok = False; break
            par[a] = b
        if ok:
            c = sum(edges[k][2] for k in combo)
            if c < best: best, found = c, [combo]
            elif c == best: found.append(combo)
    return best, found

def task5():
    banner("TASK 5 - KRUSKAL, PRIM, BORUVKA")
    print("Graph:", G5)
    print("\nKruskal:"); mk, ck = kruskal(V5, G5); print("  MST", [(u, v) for u, v, _ in mk], "cost", cost(mk), ck)
    print("Prim from A:"); mp, cp = prim(V5, G5, "A"); print("  MST", [(u, v) for u, v, _ in mp], "cost", cost(mp), cp)
    print("Boruvka:"); mb, cb = boruvka(V5, G5); print("  MST", [(u, v) for u, v, _ in mb], "cost", cost(mb), cb)
    print("Unique weights -> same edge set from all three:", edge_set(mk) == edge_set(mp) == edge_set(mb))
    print("\nModified graph (B-D lowered from 9 to 7):", [e for e in G5_TIE if e[:2] == ("B", "D")])
    best, found = all_msts(V5, G5_TIE); print(f"Exhaustive check: minimum cost {best}, number of different MSTs = {len(found)}")
    valid = {frozenset(frozenset(G5_TIE[k][:2]) for k in c) for c in found}
    runs = {"Kruskal (ties: first listed)": kruskal(V5, G5_TIE, False, False)[0], "Kruskal (ties: last listed)": kruskal(V5, G5_TIE, True, False)[0],
            "Prim from A": prim(V5, G5_TIE, "A", False)[0], "Prim from G": prim(V5, G5_TIE, "G", False)[0], "Boruvka": boruvka(V5, G5_TIE, False)[0]}
    for name, m in runs.items(): print(f"  {name:<30} cost {cost(m)}  edges {sorted(''.join(sorted(e)) for e in edge_set(m))}  valid MST: {edge_set(m) in valid}")
    print("  distinct edge sets returned:", len({edge_set(m) for m in runs.values()}))

# ============================ TASK 6: TSP (HELD-KARP DP) ============================
def held_karp(cost):
    """dp[S][j] = min cost of a path that starts at the hub 0, visits exactly the locations in S (bitmask over 1..m)
       and ends at location j (j in S)."""
    m = len(cost) - 1; stats = {"states": 0, "transitions": 0}
    if m == 0: return 0, [0, 0], stats
    dp = [[INF] * m for _ in range(1 << m)]; par = [[-1] * m for _ in range(1 << m)]
    for j in range(m): dp[1 << j][j] = cost[0][j + 1]; stats["states"] += 1          # base: hub -> j
    for S in range(1, 1 << m):                                                       # subsets before supersets
        for j in range(m):
            if not (S >> j) & 1 or S == (1 << j): continue
            rest = S ^ (1 << j); stats["states"] += 1
            for k in range(m):
                if (rest >> k) & 1:
                    stats["transitions"] += 1; c = dp[rest][k] + cost[k + 1][j + 1]
                    if c < dp[S][j]: dp[S][j] = c; par[S][j] = k
    full = (1 << m) - 1
    total, last = min((dp[full][j] + cost[j + 1][0], j) for j in range(m))           # return to hub
    route, S, j = [], full, last
    while j != -1: route.append(j + 1); S, j = S ^ (1 << j), par[S][j]
    return total, [0] + route[::-1] + [0], stats

def brute_force(cost):
    m = len(cost) - 1; best = (INF, None)
    for perm in itertools.permutations(range(1, m + 1)):
        r = (0,) + perm + (0,); c = sum(cost[r[i]][r[i + 1]] for i in range(len(r) - 1))
        if c < best[0]: best = (c, r)
    return best

def make_cost(points):
    return [[round(math.dist(p, q)) for q in points] for p in points]

def task6():
    banner("TASK 6 - TSP WITH DYNAMIC PROGRAMMING")
    names = ["Hub", "A", "B", "C", "D", "E", "F"]
    pts = [(0, 0), (2, 6), (5, 2), (7, 7), (9, 3), (3, 9), (8, 10)]; cost7 = make_cost(pts)
    print("Cost matrix (Hub + 6 locations):"); print("      " + "".join(f"{n:>5}" for n in names))
    for n, row in zip(names, cost7): print(f"  {n:>3} " + "".join(f"{c:>5}" for c in row))
    total, route, st = held_karp(cost7)
    print("\nOptimal route:", " -> ".join(names[i] for i in route), "\nMinimum Travel Cost =", total, "| DP work:", st)
    small = make_cost(pts[:5]); dpc, dpr, _ = held_karp(small); bfc, bfr = brute_force(small)
    print("\n6.4 validation (hub + 4 locations): DP =", dpc, [names[i] for i in dpr], "| exhaustive (4! = 24 routes) =", bfc, [names[i] for i in bfr],
          "| match:", dpc == bfc)
    rnd = random.Random(11); checks = 0
    for _ in range(50):
        c = make_cost([(rnd.randint(0, 50), rnd.randint(0, 50)) for _ in range(6)]); checks += held_karp(c)[0] == brute_force(c)[0]
    print(f"  extra check: DP matched brute force on {checks}/50 random 5-location instances")
    print("\n6.5 state-space growth (random points, seed 5)")
    print("  n | DP states | transitions | n! routes | time (s) | peak memory (MB)")
    rnd = random.Random(5); allpts = [(rnd.randint(0, 100), rnd.randint(0, 100)) for _ in range(15)]
    for n in range(4, 14):
        c = make_cost(allpts[:n + 1]); tracemalloc.start(); t0 = time.perf_counter(); _, _, s = held_karp(c)
        el = time.perf_counter() - t0; peak = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
        print(f"  {n:>2} | {s['states']:>9} | {s['transitions']:>11} | {math.factorial(n):>9} | {el:>8.3f} | {peak:>8.2f}")

TASKS = {1: task1, 2: task2, 3: task3, 4: task4, 5: task5, 6: task6}
if __name__ == "__main__":
    for k in RUN: TASKS[k]()


TASK 1 - WORD WRAP
Text: Dynamic programming stores answers to small overlapping subproblems so each is solved once

Width 20: total penalty = 20, lines = 5
  line 1: |Dynamic programming | unused=1 penalty=1
  line 2: |stores answers to   | unused=3 penalty=9
  line 3: |small overlapping   | unused=3 penalty=9
  line 4: |subproblems so each | unused=1 penalty=1
  line 5: |is solved once      | unused=6 penalty=0
  DP states  i | word | best[i] | next line starts at word
           0 | Dynamic    |      20 | 2
           1 | programming|     100 | 2
           2 | stores     |      19 | 5
           3 | answers    |      98 | 6
           4 | to         |       1 | 7
           5 | small      |      10 | 7
           6 | overlapping|      82 | 7
           7 | subproblems|       1 | 10
           8 | so         |       9 | 12
           9 | each       |       0 | 13
          10 | is         |       0 | 13
          11 | solved     |       0 | 13
          12 | once       |       0 | 